## Obtaining Detection Metrics

#### For CBIS-DDSM and INbreast Datasets

In [4]:
import os
import cv2
import numpy as np

#'''''''''''''''''''''''''''''''''''''''''''''''''''''''#
##Change according to were you are running the code
investigation_fles_path = "C:/Users/camiz/"
running_path = investigation_fles_path + "Breast_Cancer_Investigation/"

#'''''''''''''''''''''''''''''''''''''''''''''''''''''''#
## Chose the test dataset:
#dataset = "INbreastDataset"
#IOU_THRESHOLD = 0.4
#predictions_file = running_path + "Pipeline/" + dataset + "/Results/coordinates/bbResultcoordinatesIN.txt"

dataset = "CBIS-DDSMDataset"
IOU_THRESHOLD = 0.3
predictions_file = running_path + "Pipeline/" + dataset + "/Results/coordinates/bbResultcoordinatesCBIS.txt"
#'''''''''''''''''''''''''''''''''''''''''''''''''''''''#

labels_folder = running_path + "Pipeline/" +  dataset + "/TestLabels/"
images_folder = running_path + "Pipeline/" +  dataset + "/OriginalTestImages/"
output_folder = running_path + "Pipeline/" + dataset + "/Results/metrics/"
output_file = "resultsDetectionMetrics" + dataset + ".txt"

#Chang the format of the Roboflow coordinates to [x_min, y_min, x_max, y_max]
def yolo_to_bbox(yolo_line, img_w, img_h):
    parts = yolo_line.strip().split()
    _, x_c, y_c, w, h = map(float, parts)

    x_c *= img_w
    y_c *= img_h
    w *= img_w
    h *= img_h

    x_min = x_c - w/2
    y_min = y_c - h/2
    x_max = x_c + w/2
    y_max = y_c + h/2

    return [x_min, y_min, x_max, y_max]


def calculate_iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    inter_w = max(0, xB - xA)
    inter_h = max(0, yB - yA)
    inter_area = inter_w * inter_h

    areaA = (boxA[2]-boxA[0]) * (boxA[3]-boxA[1])
    areaB = (boxB[2]-boxB[0]) * (boxB[3]-boxB[1])

    union = areaA + areaB - inter_area

    if union == 0:
        return 0.0

    return inter_area / union


def calculate_dice(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    inter_w = max(0, xB - xA)
    inter_h = max(0, yB - yA)
    inter_area = inter_w * inter_h

    areaA = (boxA[2]-boxA[0]) * (boxA[3]-boxA[1])
    areaB = (boxB[2]-boxB[0]) * (boxB[3]-boxB[1])

    if (areaA + areaB) == 0:
        return 0.0

    return (2 * inter_area) / (areaA + areaB)

# Read predictions and store in a dictionary
predictions_dict = {}
with open(predictions_file, "r") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue

        name, coords = line.split(":")
        coords = coords.strip().replace("[", "").replace("]", "")
        coords = list(map(float, coords.split(",")))
        predictions_dict[name] = coords

# Ensure output directory exists
os.makedirs(output_folder, exist_ok=True)
output_path = os.path.join(output_folder, output_file)

# Process each label file and compare with predictions
with open(output_path, "w") as out:
    for label_file in os.listdir(labels_folder):
        if not label_file.endswith(".txt"):
            continue

        label_path = os.path.join(labels_folder, label_file)

        # Obtain the corresponding image name
        base_name = label_file.split("_jpg")[0]
        image_name = base_name + ".jpg"
        image_path = os.path.join(images_folder, image_name)

        if not os.path.exists(image_path):
            continue

        img = cv2.imread(image_path)
        img_h, img_w = img.shape[:2]

        with open(label_path, "r") as lf:
            yolo_lines = lf.readlines()

        # Process each ground truth box in the label file
        for idx, yolo_line in enumerate(yolo_lines):
            gt_bbox = yolo_to_bbox(yolo_line, img_w, img_h)

            # Construct the prediction name based on the label file and index
            pred_name = f"{base_name.split('_jpg')[0]}_{idx+1}"

            if pred_name not in predictions_dict:
                # If there is no prediction for this ground truth, we can consider it a false negative
                continue

            pred_bbox = predictions_dict[pred_name]
            iou = calculate_iou(gt_bbox, pred_bbox)
            dice = calculate_dice(gt_bbox, pred_bbox)

            # Calculate TP, FP, FN based on IoU threshold
            if iou >= IOU_THRESHOLD:
                TP = 1
                FP = 0
                FN = 0
            else:
                TP = 0
                FP = 1
                FN = 1

            precision = TP / (TP + FP) if (TP + FP) > 0 else 0
            recall = TP / (TP + FN) if (TP + FN) > 0 else 0
            # mAP50 only considers if the IoU is above the threshold
            mAP50 = 1 if iou >= IOU_THRESHOLD else 0

            out.write(f"{pred_name}: [{mAP50}, {precision}, {recall}, {iou}, {dice}]\n")

print("Process completed. Metrics saved.")

Process completed. Metrics saved.


## Average Detection Metrics

#### For CBIS-DDSM and INbreast Datasets

In [ ]:
import os
import numpy as np

#'''''''''''''''''''''''''''''''''''''''''''''''''''''''#
##Change according to were you are running the code
investigation_fles_path = "C:/Users/camiz/"
running_path = investigation_fles_path + "Breast_Cancer_Investigation/"

#'''''''''''''''''''''''''''''''''''''''''''''''''''''''#
## Chose the test dataset:
#dataset = "INbreastDataset"
dataset = "CBIS-DDSMDataset"
#'''''''''''''''''''''''''''''''''''''''''''''''''''''''#

output_folder = running_path + "Pipeline/" + dataset + "/Results/metrics/"
output_file = "resultsDetectionMetrics" + dataset + ".txt"

# Ensure output directory exists
os.makedirs(output_folder, exist_ok=True)
metrics_file = os.path.join(output_folder, output_file)

mAP50_list = []
precision_list = []
recall_list = []
iou_list = []
dice_list = []

with open(metrics_file, "r") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue

        # Extract the values inside []
        values = line.split(":")[1].strip()
        values = values.replace("[", "").replace("]", "")
        mAP50, precision, recall, iou, dice = map(float, values.split(","))

        mAP50_list.append(mAP50)
        precision_list.append(precision)
        recall_list.append(recall)
        iou_list.append(iou)
        dice_list.append(dice)

# Convert to numpy for easier calculations
mAP50_mean = np.mean(mAP50_list)
precision_mean = np.mean(precision_list)
recall_mean = np.mean(recall_list)
iou_mean = np.mean(iou_list)
dice_mean = np.mean(dice_list)

# Print average metrics
print("===== DETECTION METRICS " + dataset + " =====")
print("Mean mAP50:", f"{mAP50_mean:.4f}")
print("Mean Precision:", f"{precision_mean:.4f}")
print("Mean Recall:", f"{recall_mean:.4f}")
print("Mean IoU:", f"{iou_mean:.4f}")
print("Mean Dice:", f"{dice_mean:.4f}")

===== GLOBAL METRICS CBIS-DDSMDataset =====
Mean mAP50: 0.8968
Mean Precision: 0.8968
Mean Recall: 0.8968
Mean IoU: 0.6063
Mean Dice: 0.7202


## Average Segmentation Metrics

#### For CBIS-DDSM and INbreast Datasets

In [ ]:
import numpy as np

def average_metrics(file_name):
    values = []
    with open(file_name, 'r') as file:
        lines = file.readlines()
        for line in lines:
            # Transforms to number and appends to the list
            values.append(float(line.strip()))  

    average = np.mean(values)
    median = np.median(values)
    print(f"Mean: {average:.3f}")
    print(f"Median: {median:.3f}")

#'''''''''''''''''''''''''''''''''''''''''''''''''''''''#
##Change according to were you are running the code
investigation_fles_path = 'C:/Users/camiz/'
running_path = investigation_fles_path + 'Breast_Cancer_Investigation/Pipeline/'

#'''''''''''''''''''''''''''''''''''''''''''''''''''''''#
##Choose a dataset to average the metrics on:

##INbreastDataset
#dataset = 'INbreastDataset'
#name = 'INb'

##CBIS-DDSMDataset
dataset = 'CBIS-DDSMDataset'
name = 'CBIS'

#'''''''''''''''''''''''''''''''''''''''''''''''''''''''#
##Calculate and print average metrics
print("===== SEGMENTATION METRICS " + dataset + " =====")
print("DICE:")
file_name = running_path + dataset + '/Results/metrics/DICE_CV_' + name + '.txt' 
average_metrics(file_name)

print("IoU:")
file_name = running_path + dataset + '/Results/metrics/IOU_CV_' + name + '.txt'
average_metrics(file_name)

print("HD95:")
file_name = running_path + dataset + '/Results/metrics/HD95_CV_' + name + '.txt'
average_metrics(file_name)

DICE:
Mean: 0.700
Median: 0.730
IoU:
Mean: 0.555
Median: 0.575
HD95:
Mean: 991.331
Median: 132.851


## RESULTS:
### INbreast Dataset
Total Pipeline took 219.6273 seconds to run.  
  
===== DETECTION METRICS =====  
Mean mAP50: 0.9091  
Mean Precision: 0.9091  
Mean Recall: 0.9091  
Mean IoU: 0.6800  
Mean Dice: 0.7721  
  
===== SEGMENTATION METRICS =====   
DICE:  
Mean: 0.721  
IoU:  
Mean: 0.584  
HD95:  
Mean: 535.766  
Median: 45.829  

### CBIS-DDSM Dataset
Total Pipeline took 5618.9790 seconds to run. 
   
===== DETECTION METRICS =====  
Mean mAP50: 0.8968  
Mean Precision: 0.8968  
Mean Recall: 0.8968  
Mean IoU: 0.6063  
Mean Dice: 0.7202  
  
===== SEGMENTATION METRICS =====  
DICE:  
Mean: 0.700  
IoU:  
Mean: 0.555  
HD95:  
Mean: 991.331  
Median: 132.851  

### MiniMIAS Dataset
INbreast: Total Pipeline took 706.1391 seconds to run.  
CBIS: Total Pipeline took 373.6233 seconds to run.  